In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator
 
# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#special case company
"""
"4348"
'2285', '4081', '2286', '4340', '4342', '2283', 
'2284', '1323', '4009', '4334', '4084', '4261', 
'4012', '1830', '4333', '1835', '4165', '4347', 
'2382', '2223', '4163', '4262', '4083', '4164', 
'1834', '1303', '2081', '4162', '7204', '7202', 
'2281', '1833', '4143', '4142', '4263', '1831', 
'4051', '1322', '1182', '6017', '4161', '4031', 
'4071', '4322', '4264', '4016', '4072', '4191', 
'4332', '2381', '6014', '4017', '4015', '4324', 
'6015', '4336', '4338', '4082', '4144', '4014', 
'1111', '4291', '4339', '4013', '2282', '4325', 
'1304', '7203', '4018', '2084', '7200', '2287', 
'4331', '8313', '4193', '1183', '4192', '4345', 
'4344', '6016', '2082', '2083', '6013', '3007',

# failed training
"4003","1010", "3030","8200","1150", "2020", "1140","7020",
    "8100", "2060", "1180", "3080", 
    "2150", "4260",
    "3020", "1320", "4007", "3040",
    "8280", "6070",  "2270", "3003",  

"""


"""

# done training (4 Output already)
"3090","8300","1050"
"2050", "4001",

"1030","8170","3030","1020","3010","2060", "1180"

"4250","2280","4020","3080"

"8160"

# on hold 
"1214", "8010", "3002", "3050", "8030",  "3005",
    "2230", "4280",  "4004", "8210",  "1212",
        "1010", "8100", "1120","4002", "8012",
        "4150", "8070","3004", "2080",
    "3060", "1302", "4300", "4090", "6001", "6004",
    "7040", "2310", "2300", "8230", "4200", "4100",
"2190", "2250", "1211", "8180", "2290", "2240", "4180",
"8240", "4210", "8040", "8310", "4050", "4040", "1210", 
"""


"""" 
    
    "4006","2070","6020","1301","4130","4290"

"""

# done

""""
"8300", "3090","1050","3090","2050","4001","1030","8170","3030","1020","3010","2060", "1180",
"4250","2280","4020","3080","8160","8020","4006","2070","6020","1301","4130","4290"
"""



symbols = [
    
    ]



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 4006 ──────────────────────────
  Shape : (3036, 46)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  2014-04-02 16:00:00    4006   48.888791   53.749893  48.749903   
1  2014-04-03 16:00:00    4006   54.444336   56.666554  54.027670   
2  2014-04-06 16:00:00    4006   55.555445   55.694333  52.777672   
3  2014-04-07 16:00:00    4006   53.333227   53.333227  52.222118   
4  2014-04-08 16:00:00    4006   52.499895   53.194338  51.944341   

   close_price        volume     EMA_10     EMA_20  EMA_ratio  ...  \
0    53.749893  5.115085e+06  48.414297  45.974993   1.169111  ...   
1    54.999890  5.454337e+06  49.611678  46.834507   1.174345  ...   
2    53.472115  2.075692e+06  50.313576  47.466660   1.126519  ...   
3    52.361007  1.391457e+06  50.685836  47.932788   1.092384  ...   
4    52.222118  1.331810e+06  50.965160  48.341296   1.080280  ...   

   volume_lag_3  volume_lag_4  volume_lag_5  above_ema10  above_ema20  \
0  2.370817e+06  1.18758

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # Date-based split: train = up to 2018-12-31, test = 2019 onwards
    train_df = df[df['datetime'] <= '2018-12-31']
    test_df  = df[df['datetime'] >  '2018-12-31']

    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range   : {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Train rows: {len(train_df)}  Test rows: {len(test_df)}")


4006
  Raw train UP%      : 45.02%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2014-04-10 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 1084  Test rows: 1746

2070
  Raw train UP%      : 46.00%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2004-03-03 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 3396  Test rows: 1773

6020
  Raw train UP%      : 43.09%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2004-03-03 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 3372  Test rows: 1748

1301
  Raw train UP%      : 47.00%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2011-10-23 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 1664  Test rows: 1730

4130
  Raw train UP%      : 36.80%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000


In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    "open_price", "high_price", "low_price",
    "close_price", "volume",
    "EMA_10", "EMA_20", "EMA_ratio", "MACD_hist",
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    "ATR_14", "ATR_ratio", "BB_pct", "realized_vol",
    "OBV", "OBV_momentum", "volume_ratio",
    "volume_surge", "MFI_14",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5",
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    "volume_lag_1", "volume_lag_2", "volume_lag_3", "volume_lag_4", "volume_lag_5",
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
]

TARGET_COLS = ["price_pct_change", "volume_pct_change_log", "high_pct_change", "low_pct_change"]


def _make_seqs(X, y, seq_len):
    """Simple sliding-window builder."""
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


for symbol in symbols:
    filepath = f"{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── Regime / persistence features ─────────────────────────────────────────
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── Volume target (log-diff shifted -1) ───────────────────────────────────
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)

    # ── High / Low % change targets (shifted -1) ──────────────────────────────
    df["high_pct_change"] = df["high_price"].pct_change(1).shift(-1) 
    df["low_pct_change"]  = df["low_price"].pct_change(1).shift(-1) 

    df = df.dropna().reset_index(drop=True)

    # ── Clip outliers ─────────────────────────────────────────────────────────
    for col, sigma in {
        "price_pct_change": 3,
        "volume_pct_change_log": 2, 
        "high_pct_change": 3,
        "low_pct_change": 3
        }.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # ── Date-based split: train = up to 2018-12-31, test = 2019 onwards ───────
    train_df = df[df['datetime'] <= '2018-12-31']
    test_df  = df[df['datetime'] >  '2018-12-31']

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        "X_train": X_train,  "y_train": y_train,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Train date range: {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range: {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")


── 4006 ────────────────────────────────────────────────────────────
  Rows  → train:1042  test:1703
  Seqs  → train:1012  test:1673
  Shape → X:(1012, 30, 42)  y:(1012, 4)
  Train date range: 2014-04-10 → 2018-12-30
  Test  date range: 2018-12-31 → 2026-06-08
  Scalers saved → models/4006/

── 2070 ────────────────────────────────────────────────────────────
  Rows  → train:3240  test:1735
  Seqs  → train:3210  test:1705
  Shape → X:(3210, 30, 42)  y:(3210, 4)
  Train date range: 2004-03-03 → 2018-12-30
  Test  date range: 2018-12-31 → 2026-06-08
  Scalers saved → models/2070/

── 6020 ────────────────────────────────────────────────────────────
  Rows  → train:3172  test:1720
  Seqs  → train:3142  test:1690
  Shape → X:(3142, 30, 42)  y:(3142, 4)
  Train date range: 2004-03-03 → 2018-12-30
  Test  date range: 2018-12-31 → 2026-06-08
  Scalers saved → models/6020/

── 1301 ────────────────────────────────────────────────────────────
  Rows  → train:1607  test:1700
  Seqs  → train:157

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Test: {len(test_loader)} batches")

    sample_X, sample_y, sample_w = next(iter(train_loader))
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 4)")
    print(f"  Sample w: {sample_w.shape} → (batch,)")


4006 — Train: 32 batches | Test: 53 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)

2070 — Train: 101 batches | Test: 54 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)

6020 — Train: 99 batches | Test: 53 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)

1301 — Train: 50 batches | Test: 53 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)

4130 — Train: 142 batches | Test: 56 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)

4290 — Train: 75 

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}

# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 4)
        self.skip_fc        = nn.Linear(hidden_size * 2, 4)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss ────────────────────────────────────────────────────────────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets, weights=None):
        huber = nn.HuberLoss(delta=1.0, reduction="none")(preds, targets).mean(dim=1)

        up_weight = torch.where(targets[:, 0] > 0,torch.full_like(targets[:, 0], 1.5),torch.full_like(targets[:, 0], 1.0))
        price_sign_loss  = torch.clamp(-preds[:, 0] * targets[:, 0], min=0) * targets[:, 0].abs() 
        volume_sign_loss = torch.clamp(-preds[:, 1] * targets[:, 1], min=0) * targets[:, 1].abs()
        high_sign_loss   = torch.clamp(-preds[:, 2] * targets[:, 2], min=0) * targets[:,2].abs()
        low_sign_loss    = torch.clamp(-preds[:, 3] * targets[:, 3], min=0) * targets[:,3].abs()

        pred_std  = preds[:, 0].std() + 1e-8
        tgt_std   = targets[:, 0].std() + 1e-8
        std_ratio = pred_std / tgt_std
        std_match_loss = (pred_std - tgt_std).pow(2)
        collapse_loss  = torch.clamp(1.0 - std_ratio, min=0) ** 2

        pred_up_frac = torch.sigmoid(preds[:, 0] * 3).mean()
        bias_penalty = (pred_up_frac - 0.5).pow(2) * 60.0
        price_dir_target = (targets[:, 0] > 0).float()
        bce_loss = F.binary_cross_entropy_with_logits(preds[:, 0], price_dir_target)

        per_sample = (
            huber
            + self.alpha * price_sign_loss
            + 0.3        * volume_sign_loss
            + 0.1        * high_sign_loss
            + 0.1        * low_sign_loss
            + 3.0        * std_match_loss
        )
        if weights is not None:
            per_sample = per_sample * weights

        return (
            per_sample.mean()
            + self.beta * (1.0 / std_ratio.clamp(min=0.05)).clamp(max=10.0)
            + self.gamma * collapse_loss * 5.0
            + 5.0        * bce_loss
        )
        
        
# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor
    weight_map[3] *= oversample_factor
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values ──────────────────────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "test"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["test_X_scaled"]
    val_y      = data["test_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers     = trial.suggest_int("num_layers", 2, 3)
        dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical("batch_size", [32, 64])
        seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha          = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta           = trial.suggest_float("beta", 0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.2, 0.5, step=0.1)
        gamma          = trial.suggest_float("gamma", 0.5, 3.0, step=0.5)

        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        sampler  = make_balanced_sampler(y_tr)
        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        SEEDS       = [42, 123, 456]
        seed_scores = []

        for seed_idx, seed in enumerate(SEEDS):
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = StockPctChangeBiLSTMAttention(
                input_size  = input_size,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                dropout     = dropout
            ).to(device)

            criterion = JointPredictionLoss(alpha=alpha, beta=beta, gamma=gamma)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

            EPOCHS         = 50
            PATIENCE       = 10
            best_val_score = 0.0
            patience_ctr   = 0
            seed_collapsed = False

            for epoch in range(1, EPOCHS + 1):
                model.train()
                for X_batch, y_batch, w_batch in train_ld:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    w_batch = w_batch.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch, w_batch)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                all_preds, all_actuals = [], []
                with torch.no_grad():
                    for X_batch, y_batch, _ in val_ld:
                        all_preds.append(model(X_batch.to(device)).cpu().numpy())
                        all_actuals.append(y_batch.numpy())

                if len(all_preds) == 0:
                    seed_collapsed = True
                    break

                val_preds   = np.vstack(all_preds)
                val_actuals = np.vstack(all_actuals)

                price_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 0]).astype(int),
                    np.sign(val_preds[:, 0]).astype(int)
                )
                volume_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 1]).astype(int),
                    np.sign(val_preds[:, 1]).astype(int)
                )

                price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])
                volume_corr = safe_corr(val_actuals[:, 1], val_preds[:, 1])

                pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
                pred_up_pct    = (val_preds[:, 0] > 0).mean()
                actual_up_pct  = (val_actuals[:, 0] > 0).mean()
                bias_gap       = abs(pred_up_pct - actual_up_pct)

                if epoch >= 5:
                    if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                        seed_collapsed = True
                        break
                    if pred_std_ratio < 0.20:
                        seed_collapsed = True
                        break

                balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

                val_score = (
                    0.40 * price_dir +
                    0.20 * volume_dir +
                    0.25 * max(price_corr, 0.0) +
                    0.15 * max(volume_corr, 0.0) -
                    abs(1.0 - pred_std_ratio) * 0.3 -
                    bias_gap * 3.0 -
                    balance_penalty
                )

                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_ctr   = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= PATIENCE:
                        break

            seed_scores.append(-1.0 if seed_collapsed else best_val_score)

            trial.report(float(np.mean(seed_scores)), seed_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(seed_scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

[I 2026-06-11 17:30:47,774] A new study created in memory with name: bilstm_attention_4006


Using device: cuda

Cleaning NaN values from all symbols...
  4006 train: 1042 → 1042 rows
  4006 test: 1703 → 1703 rows
  2070 train: 3240 → 3240 rows
  2070 test: 1735 → 1735 rows
  6020 train: 3172 → 3172 rows
  6020 test: 1720 → 1720 rows
  1301 train: 1607 → 1607 rows
  1301 test: 1700 → 1700 rows
  4130 train: 4549 → 4549 rows
  4130 test: 1810 → 1810 rows
  4290 train: 2408 → 2408 rows
  4290 test: 1732 → 1732 rows

═══════════════════════════════════════════════════════
  Optuna search — 4006
═══════════════════════════════════════════════════════


[I 2026-06-11 17:31:00,854] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-11 17:31:14,096] Trial 1 finished with value: -0.6666666666666666 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 1 with value: -0.6666666666666666.
[I 2026-06-11 17:31:24,478] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 1 with value: -0.6666666666666666.
[I 2026-06-11 17:31:34,595] 


  Best joint score : -0.5741
  Best trial       : 13
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.4
    lr              : 0.0012678223615617012
    batch_size      : 64
    seq_len         : 40
    alpha           : 3.0
    beta            : 1.0
    move_threshold  : 0.30000000000000004
    gamma           : 0.5

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
13      13 -0.574112                  64                  2             0.4   
48      48 -0.575073                  64                  2             0.4   
49      49 -0.575219                  64                  2             0.4   
39      39 -0.586006                  64                  2             0.4   
23      23 -0.589844                  64                  2             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
13   0.001268                 64              40           3.0         

[I 2026-06-11 17:37:54,438] Trial 0 finished with value: -0.24779257300292623 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -0.24779257300292623.
[I 2026-06-11 17:38:23,206] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -0.24779257300292623.
[I 2026-06-11 17:39:00,584] Trial 2 finished with value: -0.6361438736175989 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -0.2477925730029

...


  Best joint score : -0.5691
  Best trial       : 0
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00018410729205738696
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.9000000000000001
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
0        0 -0.569056                 128                  3             0.3   
35      35 -0.573477                 128                  3             0.3   
33      33 -0.574526                 128                  3             0.2   
39      39 -0.590473                 128                  3             0.3   
25      25 -0.592316                 128                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
0    0.000184                 32              10      

[I 2026-06-11 18:50:25,789] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-11 18:51:17,987] Trial 1 finished with value: -0.6666666666666666 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 1 with value: -0.6666666666666666.
[I 2026-06-11 18:51:54,241] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 1 with value: -0.6666666666666666.
[I 2026-06-11 18:52:32,976] 


  Best joint score : 0.1338
  Best trial       : 45
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.30000000000000004
    lr              : 0.00024630537868142784
    batch_size      : 64
    seq_len         : 30
    alpha           : 7.0
    beta            : 1.0
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
45      45  0.133780                  64                  2             0.3   
39      39  0.112116                  64                  2             0.3   
41      41  0.110611                  64                  2             0.3   
37      37  0.074443                  64                  2             0.3   
33      33  0.072928                  64                  2             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
45   0.000246                 64              30           7.0         

[I 2026-06-11 19:24:34,446] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-11 19:25:07,742] Trial 1 finished with value: -0.6241273488975253 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 1 with value: -0.6241273488975253.
[I 2026-06-11 19:25:40,110] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 1 with value: -0.6241273488975253.
[I 2026-06-11 19:26:34,335] 


  Best joint score : 0.1815
  Best trial       : 15
  Best params:
    hidden_size     : 64
    num_layers      : 3
    dropout         : 0.2
    lr              : 0.0010436037734380145
    batch_size      : 32
    seq_len         : 30
    alpha           : 3.0
    beta            : 0.4
    move_threshold  : 0.30000000000000004
    gamma           : 1.5

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
15      15  0.181517                  64                  3             0.2   
46      46  0.164419                 128                  2             0.3   
3        3  0.139891                  64                  3             0.3   
49      49  0.136688                 128                  2             0.3   
48      48  0.129200                 128                  3             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
15   0.001044                 32              30           3.0          

In [ ]:
# Training 
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import time

trained_models = {}

def compute_joint_score(price_dir, volume_dir, price_corr, volume_corr,
                          pred_up_pct, std_ratio):
      bias_gap        = abs(pred_up_pct - 0.5)
      std_penalty     = abs(1.0 - std_ratio) * 0.3
      balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0
      return (
          0.40 * price_dir +
          0.20 * volume_dir +
          0.25 * max(price_corr,  0.0) +
          0.15 * max(volume_corr, 0.0) -
          std_penalty - bias_gap * 3.0 - balance_penalty
      )


def compute_val_metrics(model, loader, device):
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )
    volume_dir = balanced_accuracy_score(
        np.sign(actuals[:, 1]).astype(int),
        np.sign(preds[:, 1]).astype(int)
    )

    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])
    volume_corr = safe_corr(actuals[:, 1], preds[:, 1])

    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0

    joint_score = (
        0.35 * price_dir               +
        0.25 * volume_dir              +
        0.25 * max(price_corr,  0.0)   +
        0.15 * max(volume_corr, 0.0)   -
        std_penalty                    -
        collapse_penalty
    )

    return joint_score, price_dir, volume_dir, price_corr, volume_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    sampler      = make_balanced_sampler(y_train_f)
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),
        batch_size=best["batch_size"],
        sampler=sampler,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    WARMUP_EPOCHS    = 10
    BASE_ALPHA       = best["alpha"]
    BASE_BETA        = best["beta"]
    BASE_GAMMA       = best.get("gamma", 1.0)
    optimizer        = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler        = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "joint_score": [], "price_dir": [], "price_corr": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | {'VDir':>6} | "
          f"{'Pr':>6} | {'Vr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)
    
    training_state = type('State', (), {})() 
    
    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = 0.3 + 0.7 * min(1.0, epoch / WARMUP_EPOCHS) 
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = BASE_BETA,
            gamma = BASE_GAMMA
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        joint_score, p_dir, v_dir, p_corr, v_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "f"{joint_score:>7.4f} | {p_dir:>6.3f} | {v_dir:>6.3f} | "f"{p_corr:>6.3f} | {v_corr:>6.3f} | {pred_up_pct:>4.0%} | "f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection with restart ───────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if pred_up_pct > 0.85 or pred_up_pct < 0.15 or std_ratio < 0.20:
                consecutive_collapse = getattr(training_state, 'collapse_ctr', 0) + 1
                training_state.collapse_ctr = consecutive_collapse
                if consecutive_collapse >= 5:
                    print(f"\n  Collapse detected at epoch {epoch} "
                            f"(up%={pred_up_pct:.0%}, std_r={std_ratio:.2f}) — resetting weights")
                    model.apply(lambda m: m.reset_parameters()
                                if hasattr(m, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"]
                    training_state.collapse_ctr = 0
            else:
                training_state.collapse_ctr = 0

        scheduler.step(joint_score)

        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, v_dir={v_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — 4006
════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0012678223615617012, 'batch_size': 64, 'seq_len': 40, 'alpha': 3.0, 'beta': 1.0, 'move_threshold': 0.30000000000000004, 'gamma': 0.5}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |   VDir |     Pr |     Vr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 |  9.89137 | 17.68586 |  0.2356 |  0.502 |  0.497 |  0.001 | -0.028 |  84% |  0.68 |  0.4s
         ✓ Saved  (p_dir=0.502, v_dir=0.497, p_corr=0.001, up%=84%, std_r=0.68)
     2 |  8.88914 | 17.98380 |  0.2425 |  0.503 |  0.500 |  0.006 | -0.007 |  85% |  0.70 |  0.4s
         ✓ Saved  (p_dir=0.503, v_dir=0.500, p_corr=0.006, up%=85%, std_r=0.70)
     3 |  8.75716 | 17.10571 |  0.2486 |  0.504 |  0.504 | -0.006 | -0.004 |  

DIAGNOSTIC

In [8]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]
    volume_preds_pct   = np.expm1(all_preds[:, 1])   * 100
    volume_actuals_pct = np.expm1(all_actuals[:, 1]) * 100
    
    high_preds_pct     = all_preds[:, 2] * 100
    high_actuals_pct   = all_actuals[:, 2] * 100
    low_preds_pct      = all_preds[:, 3] * 100
    low_actuals_pct    = all_actuals[:, 3] * 100

    # ── Metrics ───────────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
    evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")
    evaluate(high_actuals_pct,   high_preds_pct,   "High % Change")   # add
    evaluate(low_actuals_pct,    low_preds_pct,    "Low % Change") 

    # ── Save predictions ──────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
        "actual_volume_pct"   : volume_actuals_pct,
        "predicted_volume_pct": volume_preds_pct,
        "actual_high_pct"     : high_actuals_pct,
        "predicted_high_pct"  : high_preds_pct,
        "actual_low_pct"      : low_actuals_pct,
        "predicted_low_pct"   : low_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 4006
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 1.7545%
  RMSE            : 2.2100%
  Pearson r       : 0.0102
  Pred std        : 1.5626  Actual std: 1.5364
  Within ±1%      : 36.2%
  Within ±2%      : 63.4%
  Directional Acc : 47.56%

  [Volume % Change]
  MAE             : 56.2371%
  RMSE            : 80.5314%
  Pearson r       : -0.0075
  Pred std        : 5.1622  Actual std: 78.9150
  Within ±1%      : 1.3%
  Within ±2%      : 2.9%
  Directional Acc : 49.79%

  [High % Change]
  MAE             : 1.2789%
  RMSE            : 1.6653%
  Pearson r       : 0.0515
  Pred std        : 0.3557  Actual std: 1.5255
  Within ±1%      : 49.3%
  Within ±2%      : 79.9%
  Directional Acc : 53.82%

  [Low % Change]
  MAE             : 1.0910%
  RMSE            : 1.4876%
  Pearson r       : -0.0139
  Pred std        : 0.1257  Actual std: 1.4801
  Wit

In [9]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  4006 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0783   std: 1.5369
  Predicted mean : 0.2829   std: 1.5631

  Overall Dir Acc : 47.56%
  When actual UP  : 75.48%  (722 samples)
  When actual DOWN: 26.14%  (941 samples)

  % predicted UP  : 74.56%
  % actual UP     : 43.42%

  Dir Acc by Move Size:
       <0.5% moves : 45.51%  (490 samples)
      0.5-1% moves : 51.31%  (419 samples)
        1-2% moves : 46.25%  (480 samples)
        2-5% moves : 47.74%  (266 samples)
         >5% moves : 50.00%  (8 samples)

═══════════════════════════════════════════════════════
  2070 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0640   std: 1.6559
  Predicted mean : -0.3440   std: 1.6840

  Overall Dir Acc : 49.85%
  When actual UP  : 45.12%  (851 samples)
  When actual DOWN: 54.57%  (854 samples)

  % predicted UP  : 45.28%
  % 